# Python 与 Word

## 1. 课程目标
* 认识 python-docx 库的作用与安装方法
* 掌握创建 Word、写入文本、段落、标题、表格、图片的基本 API
* 能够批量生成结构一致、内容不同的 Word 报告
* 学会设置字体、段落样式、页眉页脚，实现排版自动化

## 2. python-docx 简介
python-docx 是官方维护的第三方库，专门用于「创建」和「修改」Word 2007+ 的 .docx 文件，无需安装 Microsoft Office，跨平台运行。

| 功能   | 关键对象 / 方法                          |
| ---- | ---------------------------------- |
| 新建文档 | `Document()`                       |
| 标题   | `add_heading(text, level)`         |
| 段落   | `add_paragraph(text)`              |
| 表格   | `add_table(rows, cols, style)`     |
| 图片   | `add_picture(path, width, height)` |
| 样式   | `paragraph.style`, `run.font`      |
| 页眉页脚 | `section.header`, `section.footer` |


## 3.环境准备

In [ ]:
!pip install python-docx

## 4. 基础操作演练

### 4.1 创建并保存最简单的 Word 文档

In [ ]:
from docx import Document

doc = Document()
doc.add_heading('Python 办公自动化报告', 0)
doc.add_paragraph('这是用 python-docx 生成的第一份 Word 文档。')
doc.save('demo.docx')

### 4.2 写入多级标题与样式

In [ ]:
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()

# 一级标题
doc.add_heading('一、项目背景', level=1)

# 正文段落，并设置字体
p = doc.add_paragraph()
run = p.add_run('本项目旨在利用 Python 实现办公自动化，减少重复劳动。')
run.font.name = '微软雅黑'
run.font.size = Pt(12)

# 二级标题
doc.add_heading('二、关键发现', level=2)
p2 = doc.add_paragraph()
p2.add_run('销售额同比增长 18%，主要来源于华东区域。').bold = True
p2.alignment = WD_ALIGN_PARAGRAPH.LEFT

doc.save('demo2.docx')

### 4.3 插入表格并填充数据

In [ ]:
# 准备数据
data = [
    ['姓名', '语文', '数学', '英语'],
    ['张三', 85, 90, 88],
    ['李四', 78, 82, 80],
    ['王五', 92, 88, 85]
]

doc = Document()
doc.add_heading('学生成绩表', level=1)

# 创建 4 行 4 列表格
table = doc.add_table(rows=4, cols=4, style='Light Shading Accent 1')

# 填充单元格
for row_idx, row_data in enumerate(data):
    for col_idx, value in enumerate(row_data):
        table.cell(row_idx, col_idx).text = str(value)

# 统一表头加粗
for cell in table.rows[0].cells:
    cell.paragraphs[0].runs[0].font.bold = True

doc.save('成绩表.docx')

### 4.4 插入图片

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# 示例数据：随机生成12个月的销售额
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sales = np.random.randint(100, 500, size=12)  # 随机生成12个100到500之间的整数

# 创建 DataFrame
sales_data = pd.DataFrame({'月份': months, '销售额': sales})

# 定义图片存储路径
results_path = "/home/jovyan/work/results/png/"
os.makedirs(results_path, exist_ok=True)  # 确保路径存在

# 创建图表
plt.figure(figsize=(10, 6))
plt.plot(months, sales, marker='o', linestyle='-', color='b', label='月度销售额')

# 添加标题和标签
plt.title('月度销售额趋势', fontsize=14)
plt.xlabel('月份', fontsize=12)
plt.ylabel('销售额', fontsize=12)

# 添加图例
plt.legend()

# 添加网格线
plt.grid(True)

# 保存图表到指定路径
chart_path = os.path.join(results_path, 'Monthly_Sales_Trend.png')
plt.savefig(chart_path)
plt.close()

# 创建一个 Excel 写入器
with pd.ExcelWriter('Monthly_Sales_Report.xlsx') as writer:
    # 将数据写入 Excel
    sales_data.to_excel(writer, sheet_name='月度销售额数据', index=False)
    
    # 获取当前工作表
    worksheet = writer.sheets['月度销售额数据']
    
    # 将图表插入到 Excel 中
    worksheet.insert_image('E2', chart_path)

print("报表生成完成！")

### 4.5 设置页眉页脚

In [ ]:
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()
section = doc.sections[0]

# 页眉
header = section.header
header_para = header.paragraphs[0]
header_para.text = "内部资料  严禁外传"
header_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

# 页脚
footer = section.footer
footer_para = footer.paragraphs[0]
footer_para.text = "© 2025 公司财务部"
footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.save('带页眉页脚.docx')

## 5. 综合案例：批量生成员工绩效报告

需求：根据 Excel 中的员工数据，为每人自动生成一份 Word 绩效报告。


### 5.1 获取数据

我们这里进行数据的随机生成。

In [ ]:
import pandas as pd
import os

import pandas as pd

# 示例数据
data = {
    '姓名': ['张三', '李四', '王五', '赵六', '孙七'],
    '部门': ['销售部', '技术部', '市场部', '人力资源部', '财务部'],
    '绩效等级': ['A', 'B', 'A', 'C', 'B'],
    '奖金': [5000, 3000, 5000, 2000, 3000]
}

# 创建 DataFrame
df = pd.DataFrame(data)

# 保存到 Excel 文件
df.to_excel('员工绩效.xlsx', index=False)
df = pd.read_excel('员工绩效.xlsx')  # 列：姓名, 部门, 绩效等级, 奖金
os.makedirs('绩效报告', exist_ok=True)

### 5.2 批量生成报告

In [ ]:
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

for _, row in df.iterrows():
    doc = Document()
    
    # 标题
    title = doc.add_heading(f"{row['姓名']} 2025 年绩效报告", 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # 基础信息
    doc.add_paragraph(f"部门：{row['部门']}")
    doc.add_paragraph(f"绩效等级：{row['绩效等级']}")
    doc.add_paragraph(f"绩效奖金：{row['奖金']} 元")
    
    # 评价语
    p = doc.add_paragraph()
    p.add_run("综合评价：").bold = True
    if row['绩效等级'] == 'A':
        p.add_run("表现优秀，超额完成年度目标，继续保持！")
    else:
        p.add_run("仍需努力，争取下一年度取得更好成绩。")
    
    # 签名区
    doc.add_paragraph("\n\n部门负责人签字：_____________")
    doc.add_paragraph("日期：" + pd.Timestamp.now().strftime('%Y-%m-%d'))
    
    # 保存
    doc.save(f"绩效报告/{row['姓名']}_绩效报告.docx")

print(f"已生成 {len(df)} 份 Word 报告。")

删除所有的生成的文件

In [ ]:
import os
import shutil

# 定义要删除的文件列表
files_to_delete = [
    '员工绩效.xlsx',
    'demo.docx',
    'demo2.docx',
    '成绩表.docx',
    '带页眉页脚.docx',
    'Monthly_Sales_Report.xlsx'
]

# 定义要删除的文件夹列表
folders_to_delete = ['绩效报告']

# 删除指定文件夹及其内容
for folder in folders_to_delete:
    try:
        # 删除文件夹中的所有内容
        shutil.rmtree(folder)
        print(f"文件夹 {folder} 及其内容已删除。")
    except FileNotFoundError:
        print(f"文件夹 {folder} 不存在，跳过删除。")
    except Exception as e:
        print(f"删除文件夹 {folder} 时出错：{e}")
# 删除指定文件
for file in files_to_delete:
    try:
        os.remove(file)
        print(f"文件 {file} 已删除。")
    except FileNotFoundError:
        print(f"文件 {file} 不存在，跳过删除。")
    except Exception as e:
        print(f"删除文件 {file} 时出错：{e}")

# 删除绩效报告文件夹中的所有文件
folder_to_clean = "绩效报告"
if os.path.exists(folder_to_clean) and os.path.isdir(folder_to_clean):
    for file_name in os.listdir(folder_to_clean):
        file_path = os.path.join(folder_to_clean, file_name)
        try:
            os.remove(file_path)
            print(f"文件 {file_path} 已删除。")
        except Exception as e:
            print(f"删除文件 {file_path} 时出错：{e}")
else:
    print(f"文件夹 {folder_to_clean} 不存在，无需清理。")

print("所有指定文件已尝试删除。")